<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the libraries required to build a conversational LangChain workflow with memory and initialize the language model. 

- `InMemoryChatMessageHistory` stores the conversation as a list of user and AI messages. It keeps everything in memory for the duration of the session.
- `RunnableWithMessageHistory` handles chat history automatically for any runnable chain, so you don't need to manually record messages yourself.

These imports enable building conversational AI systems that remember past interactions.  

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key. 

In [2]:
# Import required modules
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

import os
os.environ["OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]


# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.5
)

**Note:** Be sure to run the Setup cell above before beginning with the checkpoints! 

#### Checkpoint 1/4: Creating the Session History Function 

A dictionary named `store` and a function named `get_session_history()` are defined to manage chat history across different sessions. The function takes a `session_id` parameter and returns the corresponding chat history from the `store`. 

Check if the `session_id` exists in the `store` dictionary. If it doesn't exist, create a new `InMemoryChatMessageHistory` object and add it to the `store` with the `session_id` as the key.

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [3]:
# Dictionary to store conversation histories for different sessions
store = {}

def get_session_history(session_id: str):
    """Retrieve or create a chat history for a given session."""
    ## YOUR SOLUTION HERE ##
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

#### Checkpoint 2/4: Creating a Conversational Prompt Template

Use the `ChatPromptTemplate` class to create a prompt template named `prompt` that includes conversation history. This template includes three components: 

- Add a system message that defines the assistant's role and behavior. 
- Add a `MessagesPlaceholder` with `variable_name` set to `"history"` for storing past conversation messages.
- Add a human message template with an input placeholder for the current user message. 

Create a `base_chain` that connects the prompt template, the language model, and a string output parser so each component passes its output to the next. 

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [4]:
# Create a prompt template that includes prior conversation history
## YOUR SOLUTION HERE ##
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
    "User: {input}",
    "Assistant:"
])

# Build the base conversation chain
## YOUR SOLUTION HERE ##
base_chain = prompt | llm | StrOutputParser()

#### Checkpoint 3/4: Wrapping the Chain With Message History  

At this point, we have message storage in place and a prompt that accepts conversation history.
Now, use `RunnableWithMessageHistory` to wrap the conversation chain and enable automatic history handling. This ensures that stored messages are retrieved before each request and updated after each response.

Save the wrapped chain in a variable named `conversation_chain`. 

Pass the following arguments: 
- `base_chain` as the runnable to wrap 
- `get_session_history` to retrieve or create chat history for each session 
- `input_messages_key` set to `"input"` to indicate where the user’s current message is provided 
- `history_messages_key` set to `"history"` to indicate where past conversation messages are stored

Next, define a session configuration by creating a nested dictionary with a `"configurable"` key that contains a `"session_id"` set to `"user-101"`. Assign this configuration to a variable named `session_config`. This determines which conversation history is used when invoking the chain. 

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [5]:
# Wrap with RunnableWithMessageHistory for automatic history management
## YOUR SOLUTION HERE ##
conversation_chain = RunnableWithMessageHistory(base_chain, get_session_history, input_key="input", output_key="history")

# Define session configuration
## YOUR SOLUTION HERE ##
session_config = {"configurable":  True, "session_id": "default_session"}

/Users/thomasmusieracki/Desktop/LangChain/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


#### Running a Multi-Turn Conversation

Let's run a 3-turn conversation where we introduce ourselves, share our project, and mention our methods. Each turn builds on the previous ones, and the assistant should remember what we've discussed. We'll print both the user input and assistant's response after each turn to see the full conversation flow.

In [7]:
# Turn 1
response = conversation_chain.invoke(
    {"input": "Hi! My name is Alex and I work as a data scientist."},
    config=session_config
)
print("User: Hi! My name is Alex and I work as a data scientist.")
print(f"Assistant: {response}\n")

# Turn 2
response = conversation_chain.invoke(
    {"input": "I'm working on a project about predicting customer churn."},
    config=session_config
)
print("User: I'm working on a project about predicting customer churn.")
print(f"Assistant: {response}\n")

# Turn 3
response = conversation_chain.invoke(
    {"input": "I'm using random forests and gradient boosting."},
    config=session_config
)
print("User: I'm using random forests and gradient boosting.")
print(f"Assistant: {response}\n")

AttributeError: 'bool' object has no attribute 'copy'

#### Testing Memory Recall

Now let's test if the memory actually works by asking two questions: first about our name and role, then about our project and methods. If the assistant can answer without us repeating information, it means the conversation memory is working correctly.

In [ ]:
# Test 1: Name and role
response = conversation_chain.invoke(
    {"input": "What's my name and what do I do?"},
    config=session_config
)
print("User: What's my name and what do I do?")
print(f"Assistant: {response}\n")

# Test 2: Project details
response = conversation_chain.invoke(
    {"input": "What project am I working on and what methods am I using?"},
    config=session_config
)
print("User: What project am I working on and what methods am I using?")
print(f"Assistant: {response}\n")

#### Checkpoint 4/4: Displaying the Conversation History 

Retrieve the chat history for the session by calling the `get_session_history()` function with `"user-101"` as the session ID and store the result in a variable named `session_history`. 

The provided loop iterates through all messages in the session history, assigns a role label `"User"` or `"Assistant"` based on the message type, and prints each message with its number, role, and content. 

**Note:** Don't forget to run the cell and save the notebook before selecting "Test Work". Open the "Jupyter Help" toggle at the top of the notebook for more details.

In [ ]:
# Inspect the stored conversation history
print("=" * 70)
print("Stored Conversation History:")
print("=" * 70)

## YOUR SOLUTION HERE ##
session_history = 

for i, message in enumerate(session_history.messages, 1):
    role = "User" if message.type == "human" else "Assistant"
    content = message.content
    print(f"{i}. {role}: {content}")
    print()

<details>
<summary><strong>Note</strong></summary>
Observe that the stored conversation history contains all 10 messages from the session: 5 user messages and 5 bot responses. The chat history preserves the complete conversation flow, allowing the bot to successfully recall Alex's name, role as a data scientist, the customer churn prediction project, and the machine learning methods (random forests and gradient boosting) when asked to retrieve them. 
</details>